In [ ]:
# 1) Drive only: no tokens, no git, no Kaggle API.
#    Upload to Drive:  MyDrive/s6e10/s6e10_bundle_slim.zip  (and/or MyDrive/s6e10/s6e10_bundle.zip)
from google.colab import drive
import os
drive.mount('/content/drive')
os.environ['DRV'] = DRV = '/content/drive/MyDrive/s6e10'
os.environ['S6_DATA'] = '/content/s6e10/data'      # local disk
os.environ['S6_ART'] = f'{DRV}/artifacts'          # oof/, test/, ckpt/, log_rows/, submissions/ on Drive -> resumable
os.makedirs(os.environ['S6_ART'], exist_ok=True)
!pip install -q lightgbm pyyaml kagglehub

In [ ]:
%%bash
# 2) Unpack. Slim bundle + anonymous kagglehub download of the public OOFs (md5-checked against the pinned list);
#    if that fails, or only the full bundle was uploaded, the full bundle is used instead.
set -e
cd /content
if [ -f $DRV/s6e10_bundle_slim.zip ]; then
  unzip -oq $DRV/s6e10_bundle_slim.zip -d /content/s6e10
  if (cd /content/s6e10 && python scripts/fetch_oof_anon.py); then echo "using slim bundle + kagglehub"; exit 0; fi
fi
if [ ! -f $DRV/s6e10_bundle.zip ]; then
  echo "=================================================================================="
  echo " Anonymous OOF download FAILED (or no slim bundle found)."
  echo " -> Ask Claude for the FULL bundle, upload it to  MyDrive/s6e10/s6e10_bundle.zip ,"
  echo "    then rerun this cell. Nothing else needs to change."
  echo "=================================================================================="
  exit 1
fi
unzip -oq $DRV/s6e10_bundle.zip -d /content/s6e10 && echo "using full bundle"
du -sh /content/s6e10/data/*

In [ ]:
# 3) Phase-0 checks (pandas only, ~1 min)
!cd /content/s6e10 && python scripts/verify.py

In [ ]:
# 4) E1a (all kept) then E1b (strict: all members of authors who disclosed early stopping on the scored fold);
#    CPU, ~1-1.5 h each; rerun after a disconnect (cached members + finished folds are skipped).
!cd /content/s6e10 && python src/run.py --exp e1a_stack && python src/run.py --exp e1b_stack

In [ ]:
# 5) E1a/E1b pick (saved to artifacts/e1_pick.txt) and ONE validated submission file for it (NOT submitted)
!cd /content/s6e10 && python scripts/make_sub.py --pick-e1

In [ ]:
# 6) Does E2 earn a place in the stack? Runs only once E2 (colab_e2.ipynb) and cell 5 have finished.
import os
ART = os.environ['S6_ART']
missing = [msg for path, msg in [(f'{ART}/log_rows/e2_lgb.csv', 'E2 not done yet, rerun this cell later'),
                                 (f'{ART}/e1_pick.txt', 'E1 pick missing: run cells 4-5 first, then rerun this cell')]
           if not os.path.exists(path)]
if missing:
    print('\n'.join(missing))
else:
    !cd /content/s6e10 && python src/run.py --exp e1c_stack_e2 --base-from $S6_ART/e1_pick.txt
    !cd /content/s6e10 && python scripts/gain.py --exp e1c_stack_e2